In [ ]:
#install
!pip install torch transformers scikit-learn matplotlib

!pip install torch transformers scikit-learn matplotlib scipy

In [ ]:
# Load datasets
n = 2
while n < 3:
  from google.colab import files
  uploaded = files.upload()  # Opens a file upload prompt
  n += 1

Saving peptide250_train_with_rdkit.csv to peptide250_train_with_rdkit.csv
Saving peptide250_test_with_rdkit.csv to peptide250_test_with_rdkit.csv


In [ ]:
# Load datasets
n = 2
while n < 3:
  from google.colab import files
  uploaded = files.upload()  # Opens a file upload prompt
  n += 1

Saving METLIN_RT_lipid_test_with_rdkit.csv to METLIN_RT_lipid_test_with_rdkit.csv
Saving METLIN_RT_lipid_train_with_rdkit.csv to METLIN_RT_lipid_train_with_rdkit.csv


In [ ]:
import pandas as pd

# Load full datasets
train_df = pd.read_csv('/content/peptide250_train_with_rdkit.csv')#.sample(n=50_000)
test_df  = pd.read_csv('/content/peptide250_test_with_rdkit.csv')#.sample(n=50_000)
print(len(train_df))
print(len(test_df))

# Load lipid datasets
lipid_train_df = pd.read_csv('/content/METLIN_RT_lipid_train_with_rdkit.csv')#.sample(n=10_000)
lipid_test_df  = pd.read_csv('/content/METLIN_RT_lipid_test_with_rdkit.csv')#.sample(n=10_000)
print(len(lipid_train_df))
print(len(lipid_test_df))

200000
50000
63163
7798


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.preprocessing import MinMaxScaler

# Inputs
train_smiles = train_df['smile'].tolist()
test_smiles  = test_df['smile'].tolist()

# Targets (replace 'ccs' with 'rt')
target_cols = ['rt', 'mol_weight', 'polar_surface_area', 'h_bond_donors',
               'h_bond_acceptors', 'rotatable_bonds', 'aromatic_rings', 'heavy_atoms']
y_train_orig = train_df[target_cols].values.astype(float)
y_test_orig  = test_df[target_cols].values.astype(float)

# Min-max scaling
scaler = MinMaxScaler()
y_train = scaler.fit_transform(y_train_orig).astype(np.float32)
y_test  = scaler.transform(y_test_orig).astype(np.float32)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')

# Dataset
class PeptideDataset(Dataset):
    def __init__(self, smiles, targets, tokenizer, max_len=128):
        self.smiles = smiles
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.smiles)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.smiles[idx],
                             padding='max_length',
                             truncation=True,
                             max_length=self.max_len,
                             return_tensors='pt')
        return {
            'input_ids':     enc['input_ids'].squeeze(0),
            'attention_mask':enc['attention_mask'].squeeze(0),
            'labels':        torch.tensor(self.targets[idx], dtype=torch.float32)
        }

# Datasets and loaders
train_ds = PeptideDataset(train_smiles, y_train, tokenizer)
test_ds  = PeptideDataset(test_smiles,  y_test,  tokenizer)

batch_size = 16
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class ChemBERTaRTRegressor(nn.Module):
    def __init__(self, n_outputs=8):
        super(ChemBERTaRTRegressor, self).__init__()
        # Load pre-trained ChemBERTa model
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hidden_size = self.bert.config.hidden_size  # Typically 768

        # Linear regressor head for N targets (default: 8)
        self.regressor = nn.Linear(hidden_size, n_outputs)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # Use [CLS] token
        out = self.regressor(cls_emb)  # (batch_size, n_outputs)
        return out

# Instantiate model
model = ChemBERTaRTRegressor(n_outputs=8)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Optimizer and loss
optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5)
criterion = nn.MSELoss()

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

In [ ]:
import torch.nn as nn
import torch
from sklearn.metrics import r2_score, mean_absolute_error
from transformers import AutoModel
import pandas as pd
import numpy as np  # ← needed for vstack

# Model definition
class ChemBERTaRTRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor = nn.Linear(hs, 8)  # 8 targets

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:,0,:]
        return self.regressor(cls)

# Setup
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = ChemBERTaRTRegressor().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5)
criterion = nn.MSELoss()

# Updated label names (ccs → rt)
label_names = ['rt','mol_weight','polar_surface_area','h_bond_donors',
               'h_bond_acceptors','rotatable_bonds','aromatic_rings','heavy_atoms']

def get_preds_and_truth(loader):
    model.eval()
    ys, yps = [], []
    with torch.no_grad():
        for b in loader:
            inp = b['input_ids'].to(device)
            msk = b['attention_mask'].to(device)
            out = model(inp, msk).cpu().numpy()
            ys.append(b['labels'].cpu().numpy())
            yps.append(out)
    return np.vstack(ys), np.vstack(yps)

In [ ]:
# cell7
import torch
import torch.nn as nn
from sklearn.metrics import r2_score, mean_absolute_error
from transformers import AutoModel
import pandas as pd
import numpy as np
import joblib  # <-- for saving scaler and other Python objects

# =========================
# 1) Peptide model (8 outputs, as before)
# =========================

class ChemBERTaRTRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor  = nn.Linear(hs, 8)  # 8 targets: rt + 7 descriptors

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChemBERTaRTRegressor().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5)
criterion = nn.MSELoss()

# same label names as in previous cells
label_names = ['rt', 'mol_weight', 'polar_surface_area', 'h_bond_donors',
               'h_bond_acceptors', 'rotatable_bonds', 'aromatic_rings', 'heavy_atoms']

def get_preds_and_truth_multi(loader, model_, device_):
    """For peptide multi-target regression."""
    model_.eval()
    ys, yps = [], []
    with torch.no_grad():
        for b in loader:
            inp = b['input_ids'].to(device_)
            msk = b['attention_mask'].to(device_)
            out = model_(inp, msk).cpu().numpy()
            ys.append(b['labels'].cpu().numpy())
            yps.append(out)
    return np.vstack(ys), np.vstack(yps)

# DataFrames to store peptide metrics
r2_train_df = pd.DataFrame(columns=['epoch', 'loss'] + label_names)
r2_test_df  = pd.DataFrame(columns=['epoch', 'loss'] + label_names)
mae_train_df = pd.DataFrame(columns=['epoch', 'loss'] + label_names)
mae_test_df  = pd.DataFrame(columns=['epoch', 'loss'] + label_names)

num_epochs = 15

print("===== Training on peptide data (multi-target) =====")
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0
    for b in train_loader:  # from your earlier cells
        optimizer.zero_grad()
        out = model(b['input_ids'].to(device), b['attention_mask'].to(device))
        loss = criterion(out, b['labels'].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * b['input_ids'].size(0)
    avg_loss = total_loss / len(train_ds)  # train_ds from earlier cells

    # Predictions on peptide data
    y_tr, yh_tr = get_preds_and_truth_multi(train_loader, model, device)
    y_te, yh_te = get_preds_and_truth_multi(test_loader,  model, device)

    # Inverse MinMax scaling (from cell4)
    y_tr_inv  = scaler.inverse_transform(y_tr)
    yh_tr_inv = scaler.inverse_transform(yh_tr)
    y_te_inv  = scaler.inverse_transform(y_te)
    yh_te_inv = scaler.inverse_transform(yh_te)

    r2_train_row  = {'epoch': epoch, 'loss': avg_loss}
    r2_test_row   = {'epoch': epoch, 'loss': avg_loss}
    mae_train_row = {'epoch': epoch, 'loss': avg_loss}
    mae_test_row  = {'epoch': epoch, 'loss': avg_loss}

    for i, name in enumerate(label_names):
        r2_train_row[name]  = r2_score(y_tr_inv[:, i], yh_tr_inv[:, i])
        r2_test_row[name]   = r2_score(y_te_inv[:, i], yh_te_inv[:, i])
        mae_train_row[name] = mean_absolute_error(y_tr_inv[:, i], yh_tr_inv[:, i])
        mae_test_row[name]  = mean_absolute_error(y_te_inv[:, i], yh_te_inv[:, i])

    r2_train_df.loc[len(r2_train_df)]   = r2_train_row
    r2_test_df.loc[len(r2_test_df)]     = r2_test_row
    mae_train_df.loc[len(mae_train_df)] = mae_train_row
    mae_test_df.loc[len(mae_test_df)]   = mae_test_row

    print(f"\n=== Peptide Epoch {epoch}/{num_epochs} (Loss {avg_loss:.4f}) ===")
    print(f"RT R² Train: {r2_train_row['rt']:.4f} | Test: {r2_test_row['rt']:.4f}")
    print(f"RT MAE Train: {mae_train_row['rt']:.4f} | Test: {mae_test_row['rt']:.4f}")

# Save peptide metrics (CSV)
r2_train_df.to_csv('chemberta_peptide_rt_rdkit_rsquare_train.csv', index=False)
r2_test_df.to_csv('chemberta_peptide_rt_rdkit_rsquare_test.csv', index=False)
mae_train_df.to_csv('chemberta_peptide_rt_rdkit_mae_train.csv', index=False)
mae_test_df.to_csv('chemberta_peptide_rt_rdkit_mae_test.csv', index=False)

print("\n✅ Saved peptide R² and MAE CSV files.")

# Save peptide encoder weights and scaler for future Colab runs
torch.save(model.bert.state_dict(), 'peptide_bert_encoder.pth')
joblib.dump(scaler, 'peptide_rt_scaler.pkl')

print("✅ Saved peptide encoder to 'peptide_bert_encoder.pth'.")
print("✅ Saved peptide RT scaler to 'peptide_rt_scaler.pkl'.")
print("✅ Peptide model is trained and artifacts are saved for transfer.")

===== Training on peptide data (multi-target) =====

=== Peptide Epoch 1/15 (Loss 0.0106) ===
RT R² Train: 0.6699 | Test: 0.6593
RT MAE Train: 325.5581 | Test: 331.0677

=== Peptide Epoch 2/15 (Loss 0.0076) ===
RT R² Train: 0.6989 | Test: 0.6835
RT MAE Train: 303.3966 | Test: 311.6507

=== Peptide Epoch 3/15 (Loss 0.0072) ===
RT R² Train: 0.7330 | Test: 0.7117
RT MAE Train: 279.0565 | Test: 290.3954

=== Peptide Epoch 4/15 (Loss 0.0069) ===
RT R² Train: 0.7350 | Test: 0.7061
RT MAE Train: 280.7555 | Test: 295.3996

=== Peptide Epoch 5/15 (Loss 0.0065) ===
RT R² Train: 0.7419 | Test: 0.7055
RT MAE Train: 266.7529 | Test: 284.3539

=== Peptide Epoch 6/15 (Loss 0.0062) ===
RT R² Train: 0.7541 | Test: 0.7087
RT MAE Train: 266.2919 | Test: 288.7076

=== Peptide Epoch 7/15 (Loss 0.0058) ===
RT R² Train: 0.7373 | Test: 0.6956
RT MAE Train: 296.2542 | Test: 317.0040

=== Peptide Epoch 8/15 (Loss 0.0054) ===
RT R² Train: 0.7784 | Test: 0.7236
RT MAE Train: 252.9158 | Test: 280.0262

=== Peptide

In [ ]:
# cell8 - PART 1: setup + fine-tuning on 5%, 25%, 50%
import torch
import torch.nn as nn
from sklearn.metrics import r2_score, mean_absolute_error
from transformers import AutoModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import joblib  # for saving metrics/objects

# =========================
# 2) Lipid models: ONLY RT, MinMax-scaled with peptide scaler (Option C)
# =========================

# Load lipid datasets
lipid_train_df = pd.read_csv('/content/METLIN_RT_lipid_train_with_rdkit.csv')
lipid_test_df  = pd.read_csv('/content/METLIN_RT_lipid_test_with_rdkit.csv')

# SMILES
lipid_train_smiles = lipid_train_df['smile'].tolist()
lipid_test_smiles  = lipid_test_df['smile'].tolist()

# RT (original)
rt_train = lipid_train_df['rt'].astype(float).values
rt_test  = lipid_test_df['rt'].astype(float).values

# Use SAME MinMaxScaler parameters as peptide RT (feature 0)
rt_scale = scaler.scale_[0]   # multiplicative
rt_min   = scaler.min_[0]     # additive

# Scale lipid RT with peptide scaler: X_scaled = X * scale + min
rt_train_scaled = rt_train * rt_scale + rt_min
rt_test_scaled  = rt_test  * rt_scale + rt_min

lipid_train_df['rt_scaled'] = rt_train_scaled.astype(np.float32)
lipid_test_df['rt_scaled']  = rt_test_scaled.astype(np.float32)

def inv_minmax_rt(z):
    """Inverse of peptide MinMax scaling for RT."""
    return (z - rt_min) / rt_scale

# Dataset for RT-only lipid training
class LipidRTDataset(Dataset):
    def __init__(self, smiles, rt_scaled, tokenizer, max_len=128):
        self.smiles = smiles
        self.rt_scaled = rt_scaled
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.smiles[idx],
                             padding='max_length',
                             truncation=True,
                             max_length=self.max_len,
                             return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            # shape (1,) so MSELoss works with model output (batch,1)
            'labels':         torch.tensor(self.rt_scaled[idx], dtype=torch.float32).unsqueeze(0)
        }

# One-output ChemBERTa for lipid RT
class ChemBERTaRTSingle(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor = nn.Linear(hs, 1)  # RT only

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls)

def get_preds_and_truth_rt(loader, model_, device_):
    """For RT-only lipid models in MinMax space."""
    model_.eval()
    ys, yps = [], []
    with torch.no_grad():
        for b in loader:
            inp = b['input_ids'].to(device_)
            msk = b['attention_mask'].to(device_)
            out = model_(inp, msk).cpu().numpy()   # (batch,1)
            ys.append(b['labels'].cpu().numpy())   # (batch,1)
            yps.append(out)
    # Return 1D arrays
    return np.vstack(ys).squeeze(1), np.vstack(yps).squeeze(1)


# Use 5%, 25%, 50%, 75%, 100% (full list)
percentages = [5, 25, 50, 75, 100]
fraction_values = [p / 100.0 for p in percentages]
num_epochs = 15
batch_size_lipid = 16

# Metrics: TL = transfer (peptide encoder), Base = fresh ChemBERTa
r2_train_TL = {p: [] for p in percentages}
r2_test_TL  = {p: [] for p in percentages}
mae_train_TL = {p: [] for p in percentages}
mae_test_TL  = {p: [] for p in percentages}

r2_train_Base = {p: [] for p in percentages}
r2_test_Base  = {p: [] for p in percentages}
mae_train_Base = {p: [] for p in percentages}
mae_test_Base = {p: [] for p in percentages}

print("\n===== Fitting lipid models (transfer + baseline) for 5%, 25%, 50% =====")

# First part: 5%, 25%, 50%
percentages_part1 = [5, 25, 50]
fraction_values_part1 = [p / 100.0 for p in percentages_part1]

for p, frac in zip(percentages_part1, fraction_values_part1):
    print(f"\n===== Percentage: {p}% of lipid training data =====")

    # Subsample lipid training data ONCE per percentage (same subset for TL & Base)
    n_train = max(1, int(len(lipid_train_df) * frac))  # ensure at least 1 sample
    lipid_subset = lipid_train_df.sample(n=n_train, random_state=42)

    subset_smiles = lipid_subset['smile'].tolist()
    subset_rt_scaled  = lipid_subset['rt_scaled'].values.astype(np.float32)

    # Train & test datasets/loaders
    train_dataset_lipid = LipidRTDataset(subset_smiles, subset_rt_scaled, tokenizer)
    test_dataset_lipid  = LipidRTDataset(
        lipid_test_smiles,
        lipid_test_df['rt_scaled'].values.astype(np.float32),
        tokenizer
    )

    train_loader_lipid = DataLoader(train_dataset_lipid, batch_size=batch_size_lipid, shuffle=True)
    test_loader_lipid  = DataLoader(test_dataset_lipid,  batch_size=batch_size_lipid, shuffle=False)

    # -------------------------
    # 2a) Transfer learning model (from peptide encoder)
    # -------------------------
    print("  -> Training TRANSFER model (peptide-initialized)")

    tl_model = ChemBERTaRTSingle().to(device)
    # transfer encoder weights from peptide model (trained in cell7)
    tl_model.bert.load_state_dict(model.bert.state_dict())

    optimizer_tl = torch.optim.AdamW(tl_model.parameters(), lr=2.5e-5)
    criterion_tl = nn.MSELoss()

    for epoch in range(1, num_epochs + 1):
        tl_model.train()
        total_loss_tl = 0.0
        for b in train_loader_lipid:
            optimizer_tl.zero_grad()
            out = tl_model(b['input_ids'].to(device), b['attention_mask'].to(device))  # (batch,1)
            loss = criterion_tl(out, b['labels'].to(device))
            loss.backward()
            optimizer_tl.step()
            total_loss_tl += loss.item() * b['input_ids'].size(0)
        avg_loss_tl = total_loss_tl / len(train_dataset_lipid)

        # Predictions in MinMax space
        y_tr_sc, yh_tr_sc = get_preds_and_truth_rt(train_loader_lipid, tl_model, device)
        y_te_sc, yh_te_sc = get_preds_and_truth_rt(test_loader_lipid,  tl_model, device)

        # Back to original RT
        y_tr_true = inv_minmax_rt(y_tr_sc)
        y_tr_pred = inv_minmax_rt(yh_tr_sc)
        y_te_true = inv_minmax_rt(y_te_sc)
        y_te_pred = inv_minmax_rt(yh_te_sc)

        r2_tr = r2_score(y_tr_true, y_tr_pred)
        r2_te = r2_score(y_te_true, y_te_pred)
        mae_tr = mean_absolute_error(y_tr_true, y_tr_pred)
        mae_te = mean_absolute_error(y_te_true, y_te_pred)

        r2_train_TL[p].append(r2_tr)
        r2_test_TL[p].append(r2_te)
        mae_train_TL[p].append(mae_tr)
        mae_test_TL[p].append(mae_te)

        print(f"    TL Epoch {epoch}/{num_epochs} | Loss {avg_loss_tl:.4f} | "
              f"R² Train {r2_tr:.4f} Test {r2_te:.4f} | "
              f"MAE Train {mae_tr:.2f} Test {mae_te:.2f}")

    # -------------------------
    # 2b) Baseline model: fresh ChemBERTa on same subset
    # -------------------------
    print("  -> Training BASELINE model (fresh ChemBERTa)")

    base_model = ChemBERTaRTSingle().to(device)
    optimizer_base = torch.optim.AdamW(base_model.parameters(), lr=2.5e-5)
    criterion_base = nn.MSELoss()

    for epoch in range(1, num_epochs + 1):
        base_model.train()
        total_loss_base = 0.0
        for b in train_loader_lipid:
            optimizer_base.zero_grad()
            out = base_model(b['input_ids'].to(device), b['attention_mask'].to(device))
            loss = criterion_base(out, b['labels'].to(device))
            loss.backward()
            optimizer_base.step()
            total_loss_base += loss.item() * b['input_ids'].size(0)
        avg_loss_base = total_loss_base / len(train_dataset_lipid)

        y_tr_sc, yh_tr_sc = get_preds_and_truth_rt(train_loader_lipid, base_model, device)
        y_te_sc, yh_te_sc = get_preds_and_truth_rt(test_loader_lipid,  base_model, device)

        y_tr_true = inv_minmax_rt(y_tr_sc)
        y_tr_pred = inv_minmax_rt(yh_tr_sc)
        y_te_true = inv_minmax_rt(y_te_sc)
        y_te_pred = inv_minmax_rt(yh_te_sc)

        r2_tr = r2_score(y_tr_true, y_tr_pred)
        r2_te = r2_score(y_te_true, y_te_pred)
        mae_tr = mean_absolute_error(y_tr_true, y_tr_pred)
        mae_te = mean_absolute_error(y_te_true, y_te_pred)

        r2_train_Base[p].append(r2_tr)
        r2_test_Base[p].append(r2_te)
        mae_train_Base[p].append(mae_tr)
        mae_test_Base[p].append(mae_te)

        print(f"    Base Epoch {epoch}/{num_epochs} | Loss {avg_loss_base:.4f} | "
              f"R² Train {r2_tr:.4f} Test {r2_te:.4f} | "
              f"MAE Train {mae_tr:.2f} Test {mae_te:.2f}")

print("\n✅ Finished lipid fits for 5%, 25%, 50%.")

# Save partial lipid metrics so cell9 (in another Colab) can continue from 75%, 100%
partial_metrics = {
    'percentages': percentages,
    'r2_train_TL': r2_train_TL,
    'r2_test_TL': r2_test_TL,
    'mae_train_TL': mae_train_TL,
    'mae_test_TL': mae_test_TL,
    'r2_train_Base': r2_train_Base,
    'r2_test_Base': r2_test_Base,
    'mae_train_Base': mae_train_Base,
    'mae_test_Base': mae_test_Base,
}

joblib.dump(partial_metrics, 'lipid_metrics_partial_5_25_50.pkl')
print("✅ Saved partial lipid metrics to 'lipid_metrics_partial_5_25_50.pkl'.")


===== Fitting lipid models (transfer + baseline) for 5%, 25%, 50% =====

===== Percentage: 5% of lipid training data =====
  -> Training TRANSFER model (peptide-initialized)
    TL Epoch 1/15 | Loss 0.0069 | R² Train 0.1711 Test 0.1239 | MAE Train 127.62 Test 134.50
    TL Epoch 2/15 | Loss 0.0024 | R² Train 0.3132 Test 0.2085 | MAE Train 109.09 Test 119.93
    TL Epoch 3/15 | Loss 0.0020 | R² Train 0.4529 Test 0.3079 | MAE Train 98.76 Test 113.67
    TL Epoch 4/15 | Loss 0.0017 | R² Train 0.4525 Test 0.2560 | MAE Train 97.13 Test 115.62
    TL Epoch 5/15 | Loss 0.0016 | R² Train 0.1635 Test -0.0660 | MAE Train 132.35 Test 151.39
    TL Epoch 6/15 | Loss 0.0015 | R² Train 0.6468 Test 0.3468 | MAE Train 80.54 Test 111.28
    TL Epoch 7/15 | Loss 0.0013 | R² Train 0.5799 Test 0.2529 | MAE Train 92.40 Test 124.24
    TL Epoch 8/15 | Loss 0.0011 | R² Train 0.7284 Test 0.3463 | MAE Train 72.46 Test 113.71
    TL Epoch 9/15 | Loss 0.0011 | R² Train 0.8194 Test 0.3917 | MAE Train 56.74 Test 